In [7]:
# ====================================================================
# CANCER GENOMICS & TUMOR MUTATION ANALYSIS SYSTEM (BIOINFORMATICS)
# ====================================================================
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

print("[1/3] Simulating tumor genomic sequencing cohorts...")

# 1. Programmatically generate a cohort of cancer patient genomic records
np.random.seed(42)
patient_count = 1000

# High-frequency oncogenes commonly analyzed in clinical oncology panels
oncogenes = ["TP53", "KRAS", "EGFR", "BRAF", "PIK3CA", "BRCA1", "BRCA2", "PTEN", "APC", "MYC"]
cancer_types = ["Lung Adenocarcinoma", "Colorectal Carcinoma", "Melanoma", "Breast Carcinoma", "Gglioblastoma"]

data = {
    "Patient_ID": [f"PAT-GEN-{5000 + i}" for i in range(patient_count)],
    "Cancer_Type": np.random.choice(cancer_types, patient_count, p=[0.25, 0.20, 0.15, 0.25, 0.15]),
    "Age_at_Diagnosis": np.random.normal(61, 11, patient_count).astype(int).clip(25, 90),
    "Primary_Mutated_Gene": np.random.choice(oncogenes, patient_count, p=[0.22, 0.14, 0.12, 0.10, 0.11, 0.08, 0.07, 0.06, 0.06, 0.04]),
    "Sequencing_Depth_Coverage": np.random.normal(250, 45, patient_count).astype(int).clip(50, 500),
    # Tumor Mutational Burden (TMB) = Mutations per Megabase (Mb) of sequenced DNA
    "Tumor_Mutational_Burden_TMB": np.random.exponential(scale=8.5, size=patient_count).round(2),
    "Survival_Months": np.random.uniform(3, 120, patient_count).round(1)
}

df_bio = pd.DataFrame(data)

# Inject structural genomic correlations:
# 1. Melanoma and Lung cancers typically exhibit higher TMB due to environmental mutagens (UV/Tobacco)
df_bio.loc[df_bio["Cancer_Type"] == "Melanoma", "Tumor_Mutational_Burden_TMB"] *= 2.4
df_bio.loc[df_bio["Cancer_Type"] == "Lung Adenocarcinoma", "Tumor_Mutational_Burden_TMB"] *= 1.8

# 2. Patients with high TMB and aggressive primary mutations (e.g., TP53) often face lowered clinical survival baselines
df_bio["Survival_Months"] = df_bio["Survival_Months"] - (df_bio["Tumor_Mutational_Burden_TMB"] * 1.2)
df_bio.loc[df_bio["Primary_Mutated_Gene"] == "TP53", "Survival_Months"] *= 0.75
df_bio["Survival_Months"] = df_bio["Survival_Months"].clip(2.0, 120.0).round(1)

# Categorize Clinical Response profiles
df_bio["TMB_Status"] = np.where(df_bio["Tumor_Mutational_Burden_TMB"] >= 10.0, "High TMB", "Low/Stable TMB")

print("[2/3] Processing variant allele analysis and cohorts aggregates...")
high_tmb_cohort = df_bio[df_bio["TMB_Status"] == "High TMB"]
tp53_mutants = df_bio[df_bio["Primary_Mutated_Gene"] == "TP53"]

print("\n" + "="*65)
print("     CLINICAL BIOINFORMATICS ONCOLOGY REPORT             ")
print("="*65)
print(f"🧬 Total Patient Tumors Sequenced : {patient_count} cohorts")
print(f"⚠️ High-Risk TMB Patients (>=10)   : {len(high_tmb_cohort)} cases")
print(f"📌 TP53 Driver Mutation Incidences : {len(tp53_mutants)} patients")
print(f"🔬 Average Cohort Sequencing Depth : {df_bio['Sequencing_Depth_Coverage'].mean():.1f}x coverage")
print("="*65 + "\n")

print("[3/3] Generating clinical genomic visualizations...")

# --------------------------------------------------------------------
# VISUALIZATION 1: Tumor Mutational Burden Landscape (Box Plot)
# --------------------------------------------------------------------
fig_box = px.box(
    df_bio, x="Cancer_Type", y="Tumor_Mutational_Burden_TMB", color="Cancer_Type",
    points="outliers", title="Tumor Mutational Burden (TMB) Distribution Across Tumor Classes",
    labels={"Tumor_Mutational_Burden_TMB": "TMB (Mutations / Megabase)", "Cancer_Type": "Malignancy Profile"},
    template="plotly_dark", color_discrete_sequence=px.colors.qualitative.Bold
)
fig_box.add_hline(y=10.0, line_dash="dash", line_color="#FF4B4B", annotation_text="FDA High-TMB Immunotherapy Threshold")
fig_box.show()

# --------------------------------------------------------------------
# VISUALIZATION 2: Oncogene Mutation Frequency (Bar Chart)
# --------------------------------------------------------------------
gene_counts = df_bio.groupby(["Primary_Mutated_Gene", "TMB_Status"])["Patient_ID"].count().reset_index()
fig_bar = px.bar(
    gene_counts, x="Primary_Mutated_Gene", y="Patient_ID", color="TMB_Status",
    title="Driver Oncogene Frequency Stacked by TMB Immunotherapy Profile",
    labels={"Patient_ID": "Patient Variant Volume", "Primary_Mutated_Gene": "Identified Altered Gene"},
    barmode="stack", template="plotly_dark", color_discrete_map={"High TMB": "#00D2FF", "Low/Stable TMB": "#4A5568"}
)
fig_bar.update_xaxes(categoryorder="total descending")
fig_bar.show()

# --------------------------------------------------------------------
# VISUALIZATION 3: Variant Allele Depth vs TMB Correlation (Scatter Plot)
# --------------------------------------------------------------------
fig_scatter = px.scatter(
    df_bio, x="Sequencing_Depth_Coverage", y="Tumor_Mutational_Burden_TMB",
    color="Age_at_Diagnosis", size="Survival_Months", hover_name="Patient_ID",
    title="Sequencing QC Matrix: Target Depth vs Mutational Burden Profile",
    labels={"Sequencing_Depth_Coverage": "Next-Gen Sequencing (NGS) Coverage Depth", "Tumor_Mutational_Burden_TMB": "TMB Index"},
    color_continuous_scale="Viridis", template="plotly_dark"
)
fig_scatter.show()

# --------------------------------------------------------------------
# VISUALIZATION 4: Prognostic Survival Trends (Kaplan-Meier Style Line Plot)
# --------------------------------------------------------------------
# Compute pseudo-survival curves over time intervals for high vs low TMB tracking
time_axis = np.linspace(0, 120, 20)
high_tmb_survival = [100 * np.exp(-t / 40) for t in time_axis]
low_tmb_survival = [100 * np.exp(-t / 75) for t in time_axis]

fig_survival = go.Figure()
fig_survival.add_trace(go.Scatter(x=time_axis, y=low_tmb_survival, mode='lines+markers', name='Low/Stable TMB Profile', line=dict(color='#00D2FF', width=3)))
fig_survival.add_trace(go.Scatter(x=time_axis, y=high_tmb_survival, mode='lines+markers', name='High TMB (Poor Prognosis Benchmark)', line=dict(color='#FF4B4B', width=3)))

fig_survival.update_layout(
    title="Prognostic Stratification: Pseudo Kaplan-Meier Cohort Survival Projections",
    xaxis_title="Timeline Post-Diagnosis (Months)", yaxis_title="Overall Estimated Survival Probability (%)",
    template="plotly_dark", legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99)
)
fig_survival.show()

[1/3] Simulating tumor genomic sequencing cohorts...
[2/3] Processing variant allele analysis and cohorts aggregates...

     CLINICAL BIOINFORMATICS ONCOLOGY REPORT             
🧬 Total Patient Tumors Sequenced : 1000 cohorts
⚠️ High-Risk TMB Patients (>=10)   : 398 cases
📌 TP53 Driver Mutation Incidences : 217 patients
🔬 Average Cohort Sequencing Depth : 249.6x coverage

[3/3] Generating clinical genomic visualizations...
